<a href="https://colab.research.google.com/github/AbrarYasir01/dt-cart-vs-id3/blob/main/220140.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, RocCurveDisplay
)

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
raw_url = "https://raw.githubusercontent.com/AbrarYasir01/dt-cart-vs-id3/refs/heads/main/breast-cancer.csv"
df = pd.read_csv(raw_url)
print(df.head())
print("\nColumns:", df.columns.tolist())
print("\nShape:", df.shape)
print("\nMissing values per column:\n", df.isnull().sum())

         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  radius_worst  texture_worst  perimeter_worst  area_wor

In [3]:
# Target column
target_col = "diagnosis"

# Drop id and diagnosis from features
X = df.drop(columns=["id", target_col])
y = df[target_col]

# Numeric vs categorical (here everything in X should be numeric)
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numeric columns count:", len(numeric_cols))
print("Categorical columns:", categorical_cols)

Numeric columns count: 30
Categorical columns: []


In [4]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Numeric: impute with median (will just keep values if no missing)
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical (none here, but for completeness)
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("Train size:", X_train.shape, "Test size:", X_test.shape)

Train size: (398, 30) Test size: (171, 30)


In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

cart_clf = DecisionTreeClassifier(criterion="gini", random_state=42)

cart_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", cart_clf)
])

param_grid = {
    "clf__max_depth": [3, 5, 7, None],
    "clf__min_samples_split": [2, 5, 10]
}

cart_grid = GridSearchCV(
    cart_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

cart_grid.fit(X_train, y_train)

print("Best CART params:", cart_grid.best_params_)
print("Best CART CV accuracy:", cart_grid.best_score_)

Best CART params: {'clf__max_depth': 7, 'clf__min_samples_split': 5}
Best CART CV accuracy: 0.9319936708860759


In [7]:
id3_clf = DecisionTreeClassifier(criterion="entropy", random_state=42)

id3_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", id3_clf)
])

id3_grid = GridSearchCV(
    id3_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

id3_grid.fit(X_train, y_train)

print("Best ID3 params:", id3_grid.best_params_)
print("Best ID3 CV accuracy:", id3_grid.best_score_)

Best ID3 params: {'clf__max_depth': 3, 'clf__min_samples_split': 2}
Best ID3 CV accuracy: 0.9320253164556963


In [8]:
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, RocCurveDisplay
)
import numpy as np

best_cart = cart_grid.best_estimator_
best_id3 = id3_grid.best_estimator_

# Predictions
y_pred_cart = best_cart.predict(X_test)
y_pred_id3 = best_id3.predict(X_test)

# Determine positive class: in this dataset, 'M' is malignant, 'B' benign
classes = np.unique(y_test)
print("Classes:", classes)
pos_label = "M"  # treat malignant as positive

# Probabilities for ROC/AUC
y_proba_cart = best_cart.predict_proba(X_test)[:, list(best_cart.classes_).index(pos_label)]
y_proba_id3 = best_id3.predict_proba(X_test)[:, list(best_id3.classes_).index(pos_label)]

def compute_metrics(y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label=pos_label)
    rec = recall_score(y_true, y_pred, pos_label=pos_label)
    f1 = f1_score(y_true, y_pred, pos_label=pos_label)
    auc = roc_auc_score((y_true == pos_label).astype(int), y_proba)
    return acc, prec, rec, f1, auc

metric_names = ["Accuracy", "Precision", "Recall", "F1", "AUC"]

cart_metrics = compute_metrics(y_test, y_pred_cart, y_proba_cart)
id3_metrics = compute_metrics(y_test, y_pred_id3, y_proba_id3)

print("CART metrics:", dict(zip(metric_names, cart_metrics)))
print("ID3 metrics:", dict(zip(metric_names, id3_metrics)))

Classes: ['B' 'M']
CART metrics: {'Accuracy': 0.8888888888888888, 'Precision': 0.9090909090909091, 'Recall': 0.78125, 'F1': 0.8403361344537815, 'AUC': np.float64(0.8928154205607477)}
ID3 metrics: {'Accuracy': 0.9181286549707602, 'Precision': 1.0, 'Recall': 0.78125, 'F1': 0.8771929824561403, 'AUC': np.float64(0.9342873831775701)}
